# QDSV Bridge Colab 05 - AWS Braket OpenQASM Demo

This notebook shows the public Bridge path for Amazon Braket-oriented users:

```text
problem intent -> controlled semantic spec -> Bridge OpenQASM artifact -> Braket-compatible view -> LocalSimulator -> Bridge Report
```

Bridge does not replace Amazon Braket and is not an official Amazon Braket integration. It prepares a problem-first OpenQASM artifact and reproducibility evidence that a Braket user can inspect, adapt, simulate locally, or route toward managed Braket workflows when appropriate.

## 1. Install the public SDK and Braket tools

The notebook installs `qdsv-bridge` from PyPI. The Amazon Braket SDK is used only after Bridge returns an OpenQASM artifact, so Braket remains an optional integration dependency.

In [ ]:
!pip -q install -U qdsv-bridge amazon-braket-sdk

## 2. Connect to the public Bridge API

By default this notebook uses the public demo endpoint. If you are using a private deployment, set `QDSV_BRIDGE_API_URL` and optionally `QDSV_BRIDGE_API_KEY` before creating the client.

In [ ]:
import os
from IPython.display import Markdown, display

from qdsv_bridge import QDSVBridgeClient

API_URL = os.getenv("QDSV_BRIDGE_API_URL") or None
API_KEY = os.getenv("QDSV_BRIDGE_API_KEY") or None

client = QDSVBridgeClient(api_url=API_URL, api_key=API_KEY, timeout=60)
families = client.families()

print("API status:", families.get("status"))
print("Supported families:", ", ".join(sorted(families.get("families", {}).keys())))
print("API key required for:", families.get("public_limits", {}).get("api_key_required_for"))

## 3. Declare the problem before the OpenQASM artifact

This example uses the controlled fallback family `bounded_semantic_marking`. The point is to declare the state-space role, the marking goal, the requested artifact target, and the resource limits before asking for a Braket-oriented OpenQASM artifact.

In [ ]:
spec = {
    "family": "bounded_semantic_marking",
    "bridge_mode": "build",
    "state_space": {
        "kind": "finite_candidates",
        "candidate_count": 2,
        "candidate_id": "candidate",
    },
    "signals": ["eligibility_score"],
    "prepared_candidates": [
        {"eligibility_score": 0},
        {"eligibility_score": 1},
    ],
    "goal": {
        "kind": "marking",
        "threshold": 1,
        "criteria": [
            {"signal": "eligibility_score", "importance": 1, "priority": 1}
        ],
    },
    "target": {
        "format": "qasm3",
        "backend_family": "braket",
    },
    "limits": {
        "max_qubits": 8,
        "max_depth": 160,
    },
}

spec


## 4. Build an OpenQASM artifact with Bridge

The response includes the executable artifact, canonical materialization evidence, warnings, and digests. These are the trust and reproducibility handles that Bridge adds before the artifact enters the Braket workflow.

In [ ]:
artifact_package = client.build(spec)
artifact = artifact_package["artifact"]
qasm3_source = artifact.get("content", "")

print("status:", artifact_package.get("status"))
print("family:", artifact_package.get("family"))
print("mode:", artifact_package.get("bridge_mode"))
print("artifact format:", artifact.get("format"))
print("target:", artifact_package.get("target"))
print("circuit origin:", artifact_package.get("circuit_origin"))
print("materialization evidence:", artifact_package.get("materialization_evidence"))
print("warnings:")
for warning in artifact_package.get("warnings", []):
    print("-", warning)

print("\nDigests:")
for key, value in artifact_package.get("digests", {}).items():
    print(f"- {key}: {value}")

print("\nBridge OpenQASM artifact:\n")
print(qasm3_source)

## 5. Prepare a Braket-compatible OpenQASM view

Bridge's canonical artifact uses Qiskit's standard-gate include. This notebook creates an executable Braket view by defining the gates used by the lowered circuit and mapping `cx` to Braket's `cnot` spelling.

This step is intentionally explicit: users can inspect what changed before simulation.

In [ ]:
def to_braket_openqasm(source: str) -> str:
    lines = []
    gate_definitions = [
        "gate x a { U(pi, 0, pi) a; }",
        "gate h a { U(pi/2, 0, pi) a; }",
        "gate rz(lambda) a { gphase(-lambda/2); U(0, 0, lambda) a; }",
        "gate cnot a, b { ctrl @ x a, b; }",
    ]
    for line in source.splitlines():
        stripped = line.strip()
        if stripped == "OPENQASM 3.0;":
            lines.append("OPENQASM 3;")
            lines.extend(gate_definitions)
        elif stripped == 'include "stdgates.inc";':
            continue
        elif stripped.startswith("cx "):
            lines.append(line.replace("cx ", "cnot ", 1))
        else:
            lines.append(line)
    return "\n".join(lines) + "\n"


braket_qasm3_source = to_braket_openqasm(qasm3_source)

with open("bridge_aws_braket_artifact.qasm", "w", encoding="utf-8") as f:
    f.write(qasm3_source)

with open("bridge_aws_braket_compatible.qasm", "w", encoding="utf-8") as f:
    f.write(braket_qasm3_source)

print(braket_qasm3_source)
print("Saved: bridge_aws_braket_artifact.qasm")
print("Saved: bridge_aws_braket_compatible.qasm")

## 6. Simulate locally with Amazon Braket

This step uses `LocalSimulator`, so it does not require AWS credentials, an S3 bucket, or Braket QPU access. Managed Braket devices can be considered later after inspection and any edits you need.

In [ ]:
from braket.devices import LocalSimulator
from braket.ir.openqasm import Program

program = Program(source=braket_qasm3_source)
device = LocalSimulator()
task = device.run(program, shots=1024)
result = task.result()
counts = dict(result.measurement_counts)

print("Counts:")
print(counts)

## 7. Optional managed Amazon Braket path

This notebook does not require AWS credentials. If you already use Amazon Braket, the inspected OpenQASM program can be routed through your normal Braket task workflow after you provide an AWS device and, when needed, an S3 task location.

Example outline only:

```python
# from braket.aws import AwsDevice
# from braket.ir.openqasm import Program
# device = AwsDevice("arn:aws:braket:REGION::device/qpu/PROVIDER/DEVICE")
# program = Program(source=braket_qasm3_source)
# task = device.run(program, s3_location=("your-bucket", "your-prefix"), shots=100)
# print(task.result())
```

Bridge's role is upstream of that execution step: preserve problem intent, generate an auditable artifact, and attach reproducibility metadata.

## 8. Tested environment

This demo was tested with Python 3.11, `qdsv-bridge` 0.1.7, the Amazon Braket SDK, and Amazon Braket `LocalSimulator`.

Python 3.14 is not recommended for this demo yet due to dependency compatibility issues observed in the Braket SDK / Pydantic stack.

## 9. Generate the Bridge Report

The report is the shareable evidence document. It records what was requested, what artifact was generated, what was preserved, and what warnings or limits should be reviewed before execution.

In [ ]:
report = client.report(spec, mode="build", format="markdown")
display(Markdown(report["content"]))

with open("bridge_report_aws_braket_demo.md", "w", encoding="utf-8") as f:
    f.write(report["content"])

print("Saved: bridge_report_aws_braket_demo.md")